# Cortex REST API Cost Explorer

Walk through the queries that power the Cortex REST API Cost dashboard.

**Data source:** `SNOWFLAKE.ACCOUNT_USAGE.CORTEX_REST_API_USAGE_HISTORY`  
**Billing model:** Dollars per million tokens (not credits)  
**Pricing reference:** [Service Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf) Tables 6(b) and 6(c)

> Run `deploy_all.sql` first to create the schema, pricing table, and views used in this notebook.

In [ ]:
USE WAREHOUSE SFE_CORTEX_REST_API_COST_WH;
USE SCHEMA SNOWFLAKE_EXAMPLE.CORTEX_REST_API_COST;

## 1. Raw Usage History

The `CORTEX_REST_API_USAGE_HISTORY` view in `ACCOUNT_USAGE` logs every direct REST API call.
Each row is one request. Note that there is **no credits column** -- only token counts.
The `TOKENS_GRANULAR` column is a semi-structured OBJECT with input/output breakdown.

In [ ]:
SELECT
    START_TIME,
    REQUEST_ID,
    MODEL_NAME,
    TOKENS,
    TOKENS_GRANULAR,
    USER_ID,
    INFERENCE_REGION
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_REST_API_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC
LIMIT 20;

## 2. Flatten Token Breakdown

`TOKENS_GRANULAR` is an **OBJECT** (not an ARRAY), so we use semi-structured column notation
(`:`), not `LATERAL FLATTEN`. This gives us separate input and output token counts per request.

In [ ]:
SELECT
    START_TIME,
    REQUEST_ID,
    MODEL_NAME,
    TOKENS,
    TOKENS_GRANULAR:"input"::NUMBER   AS INPUT_TOKENS,
    TOKENS_GRANULAR:"output"::NUMBER  AS OUTPUT_TOKENS,
    INFERENCE_REGION
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_REST_API_USAGE_HISTORY
WHERE START_TIME >= DATEADD('day', -30, CURRENT_TIMESTAMP())
ORDER BY START_TIME DESC
LIMIT 20;

## 3. Pricing Reference

REST API calls are billed in **dollars per million tokens**. Rates vary by model,
token type (input vs output), and inference region (regional vs global).

The `CORTEX_API_PRICING` table contains rates from the Service Consumption Table:
- **Table 6(b):** Models with prompt caching (input, output, cache_write, cache_read)
- **Table 6(c):** Models without caching (input and output only)

`REGION_CATEGORY = 'DEFAULT'` uses regional rates (higher); `'GLOBAL'` uses global rates (lower).

In [ ]:
SELECT
    MODEL_NAME,
    REGION_CATEGORY,
    INPUT_USD_PER_MTOK   AS "Input ($/M tok)",
    OUTPUT_USD_PER_MTOK  AS "Output ($/M tok)",
    CACHE_WRITE_USD_PER_MTOK AS "Cache Write ($/M tok)",
    CACHE_READ_USD_PER_MTOK  AS "Cache Read ($/M tok)",
    SOURCE_TABLE
FROM CORTEX_API_PRICING
ORDER BY SOURCE_TABLE, MODEL_NAME, REGION_CATEGORY;

## 4. Cost Per Request

The `V_API_USAGE_COSTED` view joins usage with pricing and computes the dollar cost:

```
cost = (input_tokens x input_rate / 1,000,000) + (output_tokens x output_rate / 1,000,000)
```

When a request's `INFERENCE_REGION` indicates global routing, the lower global rate is used;
otherwise the regional (DEFAULT) rate applies.

In [ ]:
SELECT
    START_TIME,
    REQUEST_ID,
    MODEL_NAME,
    INPUT_TOKENS,
    OUTPUT_TOKENS,
    INPUT_RATE    AS "Input Rate ($/M)",
    OUTPUT_RATE   AS "Output Rate ($/M)",
    INPUT_COST_USD,
    OUTPUT_COST_USD,
    TOTAL_COST_USD,
    INFERENCE_REGION
FROM V_API_USAGE_COSTED
WHERE START_TIME::DATE >= DATEADD('day', -30, CURRENT_DATE())
ORDER BY TOTAL_COST_USD DESC
LIMIT 20;

## 5. Daily Cost Trend

Aggregate by day to see spending trends. This is the same data that powers the
bar chart in the Streamlit dashboard.

In [ ]:
SELECT
    USAGE_DATE,
    REQUEST_COUNT,
    TOTAL_INPUT_TOKENS,
    TOTAL_OUTPUT_TOKENS,
    TOTAL_TOKENS,
    TOTAL_COST_USD
FROM V_DAILY_COST_SUMMARY
WHERE USAGE_DATE >= DATEADD('day', -30, CURRENT_DATE())
ORDER BY USAGE_DATE;

## 6. Cost by Model

Which models are driving the most spend? The `PCT_OF_TOTAL_COST` column shows
each model's share of total spending.

In [ ]:
SELECT
    MODEL_NAME,
    REQUEST_COUNT,
    TOTAL_INPUT_TOKENS,
    TOTAL_OUTPUT_TOKENS,
    TOTAL_COST_USD,
    PCT_OF_TOTAL_COST
FROM V_MODEL_COST_SUMMARY
ORDER BY TOTAL_COST_USD DESC;

## 7. Top Users by Spend

Identify which users are generating the most REST API cost. The `USER_ID` column
in the usage history is the internal Snowflake user identifier.

In [ ]:
SELECT
    c.USER_ID,
    u.NAME                          AS USER_NAME,
    COUNT(DISTINCT c.REQUEST_ID)    AS REQUEST_COUNT,
    SUM(c.TOKENS)                   AS TOTAL_TOKENS,
    SUM(c.TOTAL_COST_USD)           AS TOTAL_COST_USD
FROM V_API_USAGE_COSTED c
LEFT JOIN SNOWFLAKE.ACCOUNT_USAGE.USERS u
  ON c.USER_ID = u.USER_ID
WHERE c.START_TIME::DATE >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY c.USER_ID, u.NAME
ORDER BY TOTAL_COST_USD DESC
LIMIT 20;

## 8. Inference Region Distribution

See where your API calls are being routed. Regional vs. global inference has
different pricing -- global rates are typically ~10% lower.

In [ ]:
SELECT
    INFERENCE_REGION,
    MODEL_NAME,
    COUNT(DISTINCT REQUEST_ID)  AS REQUEST_COUNT,
    SUM(TOKENS)                 AS TOTAL_TOKENS,
    SUM(TOTAL_COST_USD)         AS TOTAL_COST_USD
FROM V_API_USAGE_COSTED
WHERE START_TIME::DATE >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY INFERENCE_REGION, MODEL_NAME
ORDER BY TOTAL_COST_USD DESC;

## 9. Unmatched Models (Missing Pricing)

If Snowflake adds new models or your account uses models not yet in the pricing table,
those requests will have `NULL` cost. This query finds them so you can add the rates.

In [ ]:
SELECT
    MODEL_NAME,
    INFERENCE_REGION,
    COUNT(DISTINCT REQUEST_ID) AS REQUEST_COUNT,
    SUM(TOKENS)                AS TOTAL_TOKENS
FROM V_API_USAGE_COSTED
WHERE TOTAL_COST_USD = 0
  AND TOKENS > 0
GROUP BY MODEL_NAME, INFERENCE_REGION
ORDER BY TOTAL_TOKENS DESC;

### Adding a Missing Model

If the query above returns results, look up the model's rate in the
[Service Consumption Table](https://www.snowflake.com/legal-files/CreditConsumptionTable.pdf)
(Tables 6b/6c) and insert it. Example:

In [ ]:
-- Uncomment and update with real values from the Service Consumption Table:

-- INSERT INTO CORTEX_API_PRICING
--     (MODEL_NAME, REGION_CATEGORY, INPUT_USD_PER_MTOK, OUTPUT_USD_PER_MTOK, SOURCE_TABLE)
-- VALUES
--     ('new-model-name', 'DEFAULT', 1.50, 7.50, '6b');

## 10. Monthly Summary

Roll up to month level for budget tracking and trend analysis.

In [ ]:
SELECT
    DATE_TRUNC('month', START_TIME)  AS USAGE_MONTH,
    COUNT(DISTINCT REQUEST_ID)       AS REQUEST_COUNT,
    SUM(INPUT_TOKENS)                AS TOTAL_INPUT_TOKENS,
    SUM(OUTPUT_TOKENS)               AS TOTAL_OUTPUT_TOKENS,
    SUM(TOTAL_COST_USD)              AS TOTAL_COST_USD
FROM V_API_USAGE_COSTED
GROUP BY USAGE_MONTH
ORDER BY USAGE_MONTH;

---

## Next Steps

- **Streamlit dashboard:** Open `CORTEX_REST_API_COST_APP` in Projects > Streamlit for the visual dashboard
- **Update pricing:** When Snowflake publishes new rates, INSERT/UPDATE rows in `CORTEX_API_PRICING`
- **Other Cortex billing:** This covers REST API only. For Cortex Agents, SQL AI Functions, or Intelligence, see the corresponding `ACCOUNT_USAGE` views (those bill in credits, not dollars)